In [4]:
# ============================================================
# PHASE 3 — Modèles Deep Learning (CNN1D + BiLSTM)
# Cellule 1 : Imports et chargement des données
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, warnings, joblib
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (EarlyStopping, 
                                        ModelCheckpoint,
                                        ReduceLROnPlateau)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import (f1_score, accuracy_score, 
                             precision_score, recall_score,
                             roc_auc_score)

# Fixer les seeds pour reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# Charger les données
processed_dir = r"C:\PFA_Malware\data\processed"

X_m_train = np.load(os.path.join(processed_dir, "malmem_X_train.npy"))
X_m_val   = np.load(os.path.join(processed_dir, "malmem_X_val.npy"))
X_m_test  = np.load(os.path.join(processed_dir, "malmem_X_test.npy"))
y_m_train = np.load(os.path.join(processed_dir, "malmem_y_train.npy"))
y_m_val   = np.load(os.path.join(processed_dir, "malmem_y_val.npy"))
y_m_test  = np.load(os.path.join(processed_dir, "malmem_y_test.npy"))

X_e_train = np.load(os.path.join(processed_dir, "ember_X_train.npy"))
X_e_val   = np.load(os.path.join(processed_dir, "ember_X_val.npy"))
X_e_test  = np.load(os.path.join(processed_dir, "ember_X_test.npy"))
y_e_train = np.load(os.path.join(processed_dir, "ember_y_train.npy"))
y_e_val   = np.load(os.path.join(processed_dir, "ember_y_val.npy"))
y_e_test  = np.load(os.path.join(processed_dir, "ember_y_test.npy"))

# Reshape pour DL — CNN1D et BiLSTM attendent (samples, timesteps, features)
X_m_train_dl = X_m_train.reshape(-1, X_m_train.shape[1], 1)
X_m_val_dl   = X_m_val.reshape(-1, X_m_val.shape[1], 1)
X_m_test_dl  = X_m_test.reshape(-1, X_m_test.shape[1], 1)

X_e_train_dl = X_e_train.reshape(-1, X_e_train.shape[1], 1)
X_e_val_dl   = X_e_val.reshape(-1, X_e_val.shape[1], 1)
X_e_test_dl  = X_e_test.reshape(-1, X_e_test.shape[1], 1)

print("=== Données chargées et reshapées pour DL ===")
print(f"CIC-MalMem Train : {X_m_train_dl.shape}")
print(f"CIC-MalMem Val   : {X_m_val_dl.shape}")
print(f"CIC-MalMem Test  : {X_m_test_dl.shape}")
print(f"\nEMBER Train : {X_e_train_dl.shape}")
print(f"EMBER Val   : {X_e_val_dl.shape}")
print(f"EMBER Test  : {X_e_test_dl.shape}")
print(f"\nTensorFlow version : {tf.__version__}")
print("✅ Phase 3 prête à démarrer !")

=== Données chargées et reshapées pour DL ===
CIC-MalMem Train : (40623, 28, 1)
CIC-MalMem Val   : (8705, 28, 1)
CIC-MalMem Test  : (8705, 28, 1)

EMBER Train : (480000, 200, 1)
EMBER Val   : (120000, 200, 1)
EMBER Test  : (200000, 200, 1)

TensorFlow version : 2.21.0
✅ Phase 3 prête à démarrer !


In [3]:
print(X_m_train.dtype, X_m_train.min(), X_m_train.max())
print(np.unique(y_m_train, return_counts=True))
print(X_e_train.dtype, X_e_train.min(), X_e_train.max())
print(np.unique(y_e_train, return_counts=True))

float64 -106.1199129060178 34.55132584499478
(array([0, 1]), array([20459, 20164]))
float32 -230.10014 342.47375
(array([0., 1.], dtype=float32), array([240000, 240000]))


In [3]:
# ============================================================
# Cellule 2 : CNN1D — Architecture et entraînement (version finale)
# ============================================================

# Reproductibilité — défini UNE SEULE FOIS au début
tf.random.set_seed(42)
np.random.seed(42)

# Sécurisation du dossier de sauvegarde
models_dir = r"C:\PFA_Malware\models"
os.makedirs(models_dir, exist_ok=True)

# Calcul correct du class_weight depuis les données réelles
from sklearn.utils.class_weight import compute_class_weight

def get_class_weights(y):
    """
    Calcule le class_weight depuis les labels réels.
    Si classes équilibrées → poids ~1.0 pour les deux.
    Si déséquilibre → poids plus élevé pour la classe minoritaire.
    """
    classes = np.unique(y)
    weights = compute_class_weight(
        class_weight='balanced', classes=classes, y=y)
    cw = dict(zip(classes.astype(int), weights))
    print(f"  Class weights : {cw}")
    return cw

def build_cnn1d(input_shape, filters=64, kernel_size=3,
                dropout_rate=0.3, dense_units=128, learning_rate=1e-3):
    """
    Architecture CNN1D pour détection de malware.

    Note méthodologique : CNN1D appliqué à des données tabulaires.
    Les features sont traitées comme une séquence ordonnée — choix
    expérimental justifié par la littérature récente (Raff et al., 2018 ;
    Anderson & Roth, 2018) montrant que les CNN1D capturent des
    interactions locales entre features voisines même sans ordre
    séquentiel naturel. Ce choix doit être interprété comme une
    hypothèse expérimentale et non comme une modélisation séquentielle
    stricte.
    """
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv1D(filters=filters, kernel_size=kernel_size,
                      activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2, padding='same'),
        layers.Dropout(dropout_rate),

        layers.Conv1D(filters=filters*2, kernel_size=kernel_size,
                      activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2, padding='same'),
        layers.Dropout(dropout_rate),

        layers.Flatten(),
        layers.Dense(dense_units, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

def get_callbacks(model_path):
    """
    Logique de monitoring :
    - EarlyStopping + ModelCheckpoint → val_auc (performance réelle)
    - ReduceLROnPlateau → val_loss (plus stable pour ajuster le LR,
      car la loss est une courbe lisse contrairement à l'AUC qui peut
      osciller et déclencher des réductions prématurées du LR)
    """
    return [
        EarlyStopping(monitor='val_auc', patience=10,
                      restore_best_weights=True, verbose=1,
                      mode='max'),
        ModelCheckpoint(model_path, monitor='val_auc',
                        save_best_only=True, verbose=0,
                        mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=5, min_lr=1e-6, verbose=1)
    ]

def find_best_threshold(model, X_val, y_val):
    """
    Trouve le seuil optimal sur val set.
    np.linspace utilisé pour une recherche plus fine et uniforme
    que np.arange (évite les erreurs d'arrondi flottant).
    200 points uniformément espacés sur [0.1, 0.9].
    """
    y_proba = model.predict(X_val, verbose=0).flatten()
    best_threshold, best_f1 = 0.5, 0.0
    for threshold in np.linspace(0.1, 0.9, 200):
        y_pred = (y_proba >= threshold).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        score = f1_score(y_val, y_pred, zero_division=0)
        if score > best_f1:
            best_f1        = score
            best_threshold = threshold
    print(f"  Seuil optimal : {best_threshold:.3f} | F1 val : {best_f1:.4f}")
    return best_threshold

def evaluate_dl_model(model, X_test, y_test, model_name,
                       dataset_name, threshold=0.5):
    y_proba = model.predict(X_test, verbose=0).flatten()
    y_pred  = (y_proba >= threshold).astype(int)
    return {
        'Model'    : model_name,
        'Dataset'  : dataset_name,
        'Threshold': round(threshold, 3),
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred,
                           zero_division=0), 4),
        'Recall'   : round(recall_score(y_test, y_pred,
                           zero_division=0), 4),
        'F1'       : round(f1_score(y_test, y_pred,
                           zero_division=0), 4),
        'AUC-ROC'  : round(roc_auc_score(y_test, y_proba), 4)
    }

results_dl = []

# ── input_shape dérivé des données ─────────────────────────
input_shape_malmem = (X_m_train_dl.shape[1], X_m_train_dl.shape[2])
input_shape_ember  = (X_e_train_dl.shape[1], X_e_train_dl.shape[2])
print(f"Input shape CIC-MalMem : {input_shape_malmem}")
print(f"Input shape EMBER      : {input_shape_ember}")

# ── CNN1D sur CIC-MalMem ───────────────────────────────────
print("\n" + "="*55)
print("CNN1D — CIC-MalMem")
print("="*55)

cnn_malmem = build_cnn1d(input_shape=input_shape_malmem)
cnn_malmem.summary()

print("\nCalcul class weights CIC-MalMem :")
cw_malmem = get_class_weights(y_m_train)

history_cnn_m = cnn_malmem.fit(
    X_m_train_dl, y_m_train,
    validation_data=(X_m_val_dl, y_m_val),
    epochs=50,
    batch_size=64,
    class_weight=cw_malmem,
    callbacks=get_callbacks(os.path.join(models_dir, "cnn1d_malmem.keras")),
    verbose=1
)

threshold_cnn_m = find_best_threshold(cnn_malmem, X_m_val_dl, y_m_val)
results_dl.append(evaluate_dl_model(
    cnn_malmem, X_m_test_dl, y_m_test,
    "CNN1D", "CIC-MalMem", threshold_cnn_m))
print(f"✅ CNN1D CIC-MalMem — Test F1 : {results_dl[-1]['F1']}")

# ── CNN1D sur EMBER ────────────────────────────────────────
print("\n" + "="*55)
print("CNN1D — EMBER")
print("="*55)

cnn_ember = build_cnn1d(input_shape=input_shape_ember,
                         filters=64, dense_units=128)
cnn_ember.summary()

print("\nCalcul class weights EMBER :")
cw_ember = get_class_weights(y_e_train)

history_cnn_e = cnn_ember.fit(
    X_e_train_dl, y_e_train,
    validation_data=(X_e_val_dl, y_e_val),
    epochs=50,
    batch_size=256,
    class_weight=cw_ember,
    callbacks=get_callbacks(os.path.join(models_dir, "cnn1d_ember.keras")),
    verbose=1
)

threshold_cnn_e = find_best_threshold(cnn_ember, X_e_val_dl, y_e_val)
results_dl.append(evaluate_dl_model(
    cnn_ember, X_e_test_dl, y_e_test,
    "CNN1D", "EMBER", threshold_cnn_e))
print(f"✅ CNN1D EMBER — Test F1 : {results_dl[-1]['F1']}")

print("\n🎉 CNN1D terminé !")

Input shape CIC-MalMem : (28, 1)
Input shape EMBER      : (200, 1)

CNN1D — CIC-MalMem


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 28, 64)              │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 28, 64)              │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 14, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 14, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 14, 128)             │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 14, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 7, 128)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 7, 128)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 896)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         114,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 140,673 (549.50 KB)

 Trainable params: 140,289 (548.00 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)                    │ (None, 200, 64)             │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 200, 64)             │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_2 (MaxPooling1D)       │ (None, 100, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 100, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_3 (Conv1D)                    │ (None, 100, 128)            │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 100, 128)            │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_3 (MaxPooling1D)       │ (None, 50, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 50, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 6400)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │         819,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 845,185 (3.22 MB)

 Trainable params: 844,801 (3.22 MB)

 Non-trainable params: 384 (1.50 KB)


Calcul class weights EMBER :
  Class weights : {np.int64(0): np.float64(1.0), np.int64(1): np.float64(1.0)}
Epoch 1/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 49s 26ms/step - accuracy: 0.8854 - auc: 0.9588 - loss: 0.2646 - val_accuracy: 0.9224 - val_auc: 0.9806 - val_loss: 0.1797 - learning_rate: 0.0010
Epoch 2/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 50s 27ms/step - accuracy: 0.9161 - auc: 0.9769 - loss: 0.1950 - val_accuracy: 0.9344 - val_auc: 0.9856 - val_loss: 0.1530 - learning_rate: 0.0010
Epoch 3/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 27ms/step - accuracy: 0.9260 - auc: 0.9815 - loss: 0.1741 - val_accuracy: 0.9401 - val_auc: 0.9876 - val_loss: 0.1431 - learning_rate: 0.0010
Epoch 4/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.9312 - auc: 0.9837 - loss: 0.1626 - val_accuracy: 0.9456 - val_auc: 0.9893 - val_loss: 0.1332 - learning_rate: 0.0010
Epoch 5/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.9357 - auc: 0.9856 - loss: 0.1523 - val_accuracy: 0.9483 - val_auc: 0

In [5]:
# ============================================================
# RECHARGEMENT COMPLET — À lancer après chaque restart
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, warnings, joblib
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (EarlyStopping,
                                        ModelCheckpoint,
                                        ReduceLROnPlateau)
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)
np.random.seed(42)

# ── Chemins ───────────────────────────────────────────────
processed_dir = r"C:\PFA_Malware\data\processed"
models_dir    = r"C:\PFA_Malware\models"
results_dir   = r"C:\PFA_Malware\results"
os.makedirs(models_dir, exist_ok=True)

# ── Charger données ────────────────────────────────────────
X_m_train = np.load(os.path.join(processed_dir, "malmem_X_train.npy"))
X_m_val   = np.load(os.path.join(processed_dir, "malmem_X_val.npy"))
X_m_test  = np.load(os.path.join(processed_dir, "malmem_X_test.npy"))
y_m_train = np.load(os.path.join(processed_dir, "malmem_y_train.npy"))
y_m_val   = np.load(os.path.join(processed_dir, "malmem_y_val.npy"))
y_m_test  = np.load(os.path.join(processed_dir, "malmem_y_test.npy"))

X_e_train = np.load(os.path.join(processed_dir, "ember_X_train.npy"))
X_e_val   = np.load(os.path.join(processed_dir, "ember_X_val.npy"))
X_e_test  = np.load(os.path.join(processed_dir, "ember_X_test.npy"))
y_e_train = np.load(os.path.join(processed_dir, "ember_y_train.npy"))
y_e_val   = np.load(os.path.join(processed_dir, "ember_y_val.npy"))
y_e_test  = np.load(os.path.join(processed_dir, "ember_y_test.npy"))

# ── Reshape pour CNN1D ─────────────────────────────────────
X_m_val_dl  = X_m_val.reshape(-1, X_m_val.shape[1], 1)
X_m_test_dl = X_m_test.reshape(-1, X_m_test.shape[1], 1)
X_e_val_dl  = X_e_val.reshape(-1, X_e_val.shape[1], 1)
X_e_test_dl = X_e_test.reshape(-1, X_e_test.shape[1], 1)

# ── Fonctions utilitaires ──────────────────────────────────
def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(
        class_weight='balanced', classes=classes, y=y)
    cw = dict(zip(classes.astype(int), weights))
    print(f"  Class weights : {cw}")
    return cw

def get_callbacks(model_path):
    return [
        EarlyStopping(monitor='val_auc', patience=10,
                      restore_best_weights=True, verbose=1,
                      mode='max'),
        ModelCheckpoint(model_path, monitor='val_auc',
                        save_best_only=True, verbose=0,
                        mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=5, min_lr=1e-6, verbose=1)
    ]

def find_best_threshold(model, X_val, y_val):
    y_proba = model.predict(X_val, verbose=0).flatten()
    best_threshold, best_f1 = 0.5, 0.0
    for threshold in np.linspace(0.1, 0.9, 200):
        y_pred = (y_proba >= threshold).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        score = f1_score(y_val, y_pred, zero_division=0)
        if score > best_f1:
            best_f1        = score
            best_threshold = threshold
    print(f"  Seuil optimal : {best_threshold:.3f} | F1 val : {best_f1:.4f}")
    return best_threshold

def evaluate_dl_model(model, X_test, y_test, model_name,
                       dataset_name, threshold=0.5):
    y_proba = model.predict(X_test, verbose=0).flatten()
    y_pred  = (y_proba >= threshold).astype(int)
    return {
        'Model'    : model_name,
        'Dataset'  : dataset_name,
        'Threshold': round(threshold, 3),
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred,
                           zero_division=0), 4),
        'Recall'   : round(recall_score(y_test, y_pred,
                           zero_division=0), 4),
        'F1'       : round(f1_score(y_test, y_pred,
                           zero_division=0), 4),
        'AUC-ROC'  : round(roc_auc_score(y_test, y_proba), 4)
    }

# ── Recalcul résultats CNN1D depuis modèles sauvegardés ────
print("Chargement modèles CNN1D...")
cnn_malmem = keras.models.load_model(
    os.path.join(models_dir, "cnn1d_malmem.keras"))
cnn_ember  = keras.models.load_model(
    os.path.join(models_dir, "cnn1d_ember.keras"))

print("Recalcul seuils et métriques CNN1D...")
threshold_cnn_m = find_best_threshold(cnn_malmem, X_m_val_dl, y_m_val)
threshold_cnn_e = find_best_threshold(cnn_ember,  X_e_val_dl, y_e_val)

results_dl_final = [
    evaluate_dl_model(cnn_malmem, X_m_test_dl, y_m_test,
                      "CNN1D", "CIC-MalMem", threshold_cnn_m),
    evaluate_dl_model(cnn_ember,  X_e_test_dl, y_e_test,
                      "CNN1D", "EMBER",       threshold_cnn_e)
]

print(f"\n✅ Rechargement complet terminé !")
print(f"CIC-MalMem : {X_m_train.shape} | EMBER : {X_e_train.shape}")
print(f"CNN1D résultats recalculés : {len(results_dl_final)}")

Chargement modèles CNN1D...
Recalcul seuils et métriques CNN1D...
  Seuil optimal : 0.397 | F1 val : 1.0000
  Seuil optimal : 0.478 | F1 val : 0.9641

✅ Rechargement complet terminé !
CIC-MalMem : (40623, 28) | EMBER : (480000, 200)
CNN1D résultats recalculés : 2


In [7]:
# ============================================================
# Cellule 3 : FT-Transformer — Version finale corrigée
# Gorishniy et al., NeurIPS 2021
# ============================================================

# ── Feature Tokenizer ──────────────────────────────────────
class FeatureTokenizer(layers.Layer):
    """
    Transforme chaque feature numérique en embedding de dimension d_token.
    Simplification assumée : features numériques uniquement —
    toutes nos features sont numériques après Phase 1.
    f_i → e_i = f_i * W_i + b_i
    """
    def __init__(self, n_features, d_token, **kwargs):
        super().__init__(**kwargs)
        self.n_features = n_features
        self.d_token    = d_token

    def build(self, input_shape):
        self.W = self.add_weight(
            name='tokenizer_weights',
            shape=(self.n_features, self.d_token),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='tokenizer_biases',
            shape=(self.n_features, self.d_token),
            initializer='zeros',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        return x[:, :, None] * self.W[None, :, :] + self.b[None, :, :]

    def get_config(self):
        config = super().get_config()
        config.update({
            'n_features': self.n_features,
            'd_token'   : self.d_token
        })
        return config

# ── CLS Token — couche dédiée Keras ────────────────────────
class ClsToken(layers.Layer):
    """
    Token [CLS] appris encapsulé dans une couche Keras propre.
    Utilise add_weight() pour un suivi correct par Keras :
    sauvegarde, rechargement et reproductibilité garantis.
    Conforme aux bonnes pratiques Keras et au papier original.
    """
    def __init__(self, d_token, **kwargs):
        super().__init__(**kwargs)
        self.d_token = d_token

    def build(self, input_shape):
        self.cls = self.add_weight(
            name='cls_token',
            shape=(1, 1, self.d_token),
            initializer='zeros',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls_tokens = tf.repeat(self.cls, batch_size, axis=0)
        return tf.concat([cls_tokens, x], axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({'d_token': self.d_token})
        return config

# ── Transformer Block ───────────────────────────────────────
class TransformerBlock(layers.Layer):
    """
    Bloc Transformer avec Pre-Layer Normalization (Pre-LN).
    Limites assumées et documentées :
    - Pas de positional encoding (features tabulaires sans ordre naturel)
    - n_blocks=2 et d_token modeste : compromis vitesse/performance
    - Coût quadratique en nombre de features (attention O(n²))
    """
    def __init__(self, d_token, n_heads, ffn_factor=2.0,
                 dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_token      = d_token
        self.n_heads      = n_heads
        self.ffn_factor   = ffn_factor
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        self.attention = layers.MultiHeadAttention(
            num_heads=self.n_heads,
            key_dim=self.d_token // self.n_heads,
            dropout=self.dropout_rate
        )
        self.ffn = keras.Sequential([
            layers.Dense(int(self.d_token * self.ffn_factor),
                         activation='relu'),
            layers.Dropout(self.dropout_rate),
            layers.Dense(self.d_token)
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1  = layers.Dropout(self.dropout_rate)
        self.drop2  = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, x, training=False):
        # Pre-LN + Self-Attention + Residual
        attn_out = self.attention(
            self.norm1(x), self.norm1(x), training=training)
        attn_out = self.drop1(attn_out, training=training)
        x = x + attn_out

        # Pre-LN + FFN + Residual
        ffn_out = self.ffn(self.norm2(x), training=training)
        ffn_out = self.drop2(ffn_out, training=training)
        x = x + ffn_out
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_token'     : self.d_token,
            'n_heads'     : self.n_heads,
            'ffn_factor'  : self.ffn_factor,
            'dropout_rate': self.dropout_rate
        })
        return config

# ── FT-Transformer complet ──────────────────────────────────
def build_ft_transformer(n_features, d_token=32, n_heads=4,
                          n_blocks=2, ffn_factor=2.0,
                          dropout_rate=0.1, learning_rate=1e-3):
    """
    FT-Transformer conforme à Gorishniy et al. (2021).

    Simplifications assumées et documentées :
    1. Features numériques uniquement (pas de catégorielles)
    2. Pas de positional encoding (données tabulaires sans ordre)
    3. n_blocks=2 et d_token modeste : compromis vitesse/performance
    4. Sensible aux hyperparamètres d_token, n_heads, learning_rate
    5. Coût quadratique en attention O(n_features²)

    Conformité papier :
    ✅ Feature Tokenizer par feature
    ✅ Token [CLS] appris via ClsToken layer
    ✅ Pre-LN Transformer blocks
    ✅ Classification via token [CLS] position 0
    """
    assert d_token % n_heads == 0, \
        f"d_token ({d_token}) doit être divisible par n_heads ({n_heads})"

    inputs = keras.Input(shape=(n_features,), name='features')

    # Feature Tokenizer : (batch, n_features) → (batch, n_features, d_token)
    x = FeatureTokenizer(n_features, d_token,
                         name='feature_tokenizer')(inputs)

    # Ajout token [CLS] : (batch, n_features+1, d_token)
    x = ClsToken(d_token, name='cls_token')(x)

    # Blocs Transformer
    for i in range(n_blocks):
        x = TransformerBlock(
            d_token=d_token,
            n_heads=n_heads,
            ffn_factor=ffn_factor,
            dropout_rate=dropout_rate,
            name=f'transformer_block_{i}'
        )(x)

    # Extraction token [CLS] — position 0
    cls_output = x[:, 0, :]

    # Tête de classification
    x = layers.LayerNormalization(epsilon=1e-6)(cls_output)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(1, activation='sigmoid',
                           name='classification')(x)

    model = keras.Model(inputs=inputs, outputs=outputs,
                        name='FT_Transformer')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

# ── Pipeline robuste ────────────────────────────────────────
try:
    results_cnn1d = [r for r in results_dl if r['Model'] == 'CNN1D']
except NameError:
    print("⚠️ results_dl non trouvé — recalcul des scores CNN1D...")
    results_cnn1d = []

    from tensorflow.keras.models import load_model

    cnn_malmem = load_model(
        os.path.join(models_dir, "cnn1d_malmem.keras"),
        custom_objects={}
    )
    cnn_ember = load_model(
        os.path.join(models_dir, "cnn1d_ember.keras"),
        custom_objects={}
    )

    threshold_cnn_m = find_best_threshold(cnn_malmem, X_m_val_dl, y_m_val)
    results_cnn1d.append(evaluate_dl_model(
        cnn_malmem, X_m_test_dl, y_m_test,
        "CNN1D", "CIC-MalMem", threshold_cnn_m))

    threshold_cnn_e = find_best_threshold(cnn_ember, X_e_val_dl, y_e_val)
    results_cnn1d.append(evaluate_dl_model(
        cnn_ember, X_e_test_dl, y_e_test,
        "CNN1D", "EMBER", threshold_cnn_e))

    print(f"✅ CNN1D recalculé — {len(results_cnn1d)} résultats")

results_dl_final = results_cnn1d.copy()
print(f"Résultats CNN1D conservés : {len(results_cnn1d)}")

n_features_malmem = X_m_train.shape[1]
n_features_ember  = X_e_train.shape[1]
print(f"Features CIC-MalMem : {n_features_malmem}")
print(f"Features EMBER      : {n_features_ember}")

# ── FT-Transformer CIC-MalMem ──────────────────────────────
print("\n" + "="*55)
print("FT-Transformer — CIC-MalMem")
print("="*55)

ft_malmem = build_ft_transformer(
    n_features=n_features_malmem,
    d_token=32, n_heads=4, n_blocks=2,
    dropout_rate=0.1, learning_rate=1e-3
)
ft_malmem.summary()

print("\nCalcul class weights CIC-MalMem :")
cw_malmem = get_class_weights(y_m_train)

history_ft_m = ft_malmem.fit(
    X_m_train, y_m_train,
    validation_data=(X_m_val, y_m_val),
    epochs=50, batch_size=64,
    class_weight=cw_malmem,
    callbacks=get_callbacks(
        os.path.join(models_dir, "ft_transformer_malmem.keras")),
    verbose=1
)

threshold_ft_m = find_best_threshold(ft_malmem, X_m_val, y_m_val)
results_dl_final.append(evaluate_dl_model(
    ft_malmem, X_m_test, y_m_test,
    "FT-Transformer", "CIC-MalMem", threshold_ft_m))
print(f"✅ FT-Transformer CIC-MalMem — F1 : {results_dl_final[-1]['F1']}")

# ── FT-Transformer EMBER ────────────────────────────────────
print("\n" + "="*55)
print("FT-Transformer — EMBER")
print("="*55)

ft_ember = build_ft_transformer(
    n_features=n_features_ember,
    d_token=64, n_heads=4, n_blocks=2,
    dropout_rate=0.2, learning_rate=1e-3
)
ft_ember.summary()

print("\nCalcul class weights EMBER :")
cw_ember = get_class_weights(y_e_train)

history_ft_e = ft_ember.fit(
    X_e_train, y_e_train,
    validation_data=(X_e_val, y_e_val),
    epochs=50, batch_size=512,
    class_weight=cw_ember,
    callbacks=get_callbacks(
        os.path.join(models_dir, "ft_transformer_ember.keras")),
    verbose=1
)

threshold_ft_e = find_best_threshold(ft_ember, X_e_val, y_e_val)
results_dl_final.append(evaluate_dl_model(
    ft_ember, X_e_test, y_e_test,
    "FT-Transformer", "EMBER", threshold_ft_e))
print(f"✅ FT-Transformer EMBER — F1 : {results_dl_final[-1]['F1']}")

# ── Tableau récapitulatif Phase 3 ──────────────────────────
print("\n" + "="*55)
print("TABLEAU RÉCAPITULATIF — Phase 3 DL")
print("="*55)
df_dl = pd.DataFrame(results_dl_final)
print(df_dl.to_string(index=False))

print("\n🎉 Phase 3 — FT-Transformer terminé !")

⚠️ results_dl non trouvé — recalcul des scores CNN1D...
  Seuil optimal : 0.397 | F1 val : 1.0000
  Seuil optimal : 0.478 | F1 val : 0.9641
✅ CNN1D recalculé — 2 résultats
Résultats CNN1D conservés : 2
Features CIC-MalMem : 28
Features EMBER      : 200

FT-Transformer — CIC-MalMem


Model: "FT_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ features (InputLayer)                │ (None, 28)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ feature_tokenizer (FeatureTokenizer) │ (None, 28, 32)              │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ cls_token (ClsToken)                 │ (None, 29, 32)              │              32 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block_0                  │ (None, 29, 32)              │           8,544 │
│ (TransformerBlock)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block_1                  │ (None, 29, 32)              │           8,544 │
│ (TransformerBlock)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ get_item_2 (GetItem)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer_normalization_14               │ (None, 32)                  │              64 │
│ (LayerNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_14 (Dense)                     │ (None, 64)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_26 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ classification (Dense)               │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 21,153 (82.63 KB)

 Trainable params: 21,153 (82.63 KB)

...

 Trainable params: 96,961 (378.75 KB)

 Non-trainable params: 0 (0.00 B)


Calcul class weights EMBER :
  Class weights : {np.int64(0): np.float64(1.0), np.int64(1): np.float64(1.0)}
Epoch 1/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 787s 833ms/step - accuracy: 0.8529 - auc: 0.9361 - loss: 0.3199 - val_accuracy: 0.8826 - val_auc: 0.9581 - val_loss: 0.2786 - learning_rate: 0.0010
Epoch 2/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 810s 864ms/step - accuracy: 0.8870 - auc: 0.9603 - loss: 0.2544 - val_accuracy: 0.9003 - val_auc: 0.9676 - val_loss: 0.2441 - learning_rate: 0.0010
Epoch 3/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 810s 863ms/step - accuracy: 0.8984 - auc: 0.9667 - loss: 0.2336 - val_accuracy: 0.9055 - val_auc: 0.9719 - val_loss: 0.2200 - learning_rate: 0.0010
Epoch 4/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 809s 862ms/step - accuracy: 0.9046 - auc: 0.9701 - loss: 0.2211 - val_accuracy: 0.9111 - val_auc: 0.9738 - val_loss: 0.2098 - learning_rate: 0.0010
Epoch 5/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 809s 863ms/step - accuracy: 0.9105 - auc: 0.9731 - loss: 0.2101 - val_accuracy: 0.9171 - val_auc: 0

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



938/938 ━━━━━━━━━━━━━━━━━━━━ 809s 863ms/step - accuracy: 0.9417 - auc: 0.9872 - loss: 0.1435 - val_accuracy: 0.9418 - val_auc: 0.9857 - val_loss: 0.1551 - learning_rate: 0.0010
Epoch 28/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 811s 864ms/step - accuracy: 0.9426 - auc: 0.9873 - loss: 0.1424 - val_accuracy: 0.9416 - val_auc: 0.9857 - val_loss: 0.1551 - learning_rate: 0.0010
Epoch 29/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 811s 865ms/step - accuracy: 0.9427 - auc: 0.9874 - loss: 0.1419 - val_accuracy: 0.9431 - val_auc: 0.9862 - val_loss: 0.1518 - learning_rate: 0.0010
Epoch 30/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 811s 865ms/step - accuracy: 0.9433 - auc: 0.9876 - loss: 0.1406 - val_accuracy: 0.9422 - val_auc: 0.9861 - val_loss: 0.1528 - learning_rate: 0.0010
Epoch 31/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 811s 865ms/step - accuracy: 0.9435 - auc: 0.9879 - loss: 0.1393 - val_accuracy: 0.9419 - val_auc: 0.9859 - val_loss: 0.1542 - learning_rate: 0.0010
Epoch 32/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 810s 863ms/step - accuracy

In [8]:
# ============================================================
# Cellule 4 : Sauvegarde Phase 3 finale
# ============================================================
import json, os

results_dir = r"C:\PFA_Malware\results"
models_dir  = r"C:\PFA_Malware\models"

# Sauvegarde tableau résultats
df_dl = pd.DataFrame(results_dl_final)
df_dl.to_csv(os.path.join(results_dir, "dl_results_final.csv"), index=False)
print("✅ Tableau résultats DL sauvegardé")

# Sauvegarde configurations extraites des résultats réels
config_dl = {}
for r in results_dl_final:
    key = f"{r['Model']}_{r['Dataset']}"
    config_dl[key] = {
        "threshold": r['Threshold'],
        "F1_test"  : r['F1'],
        "AUC_test" : r['AUC-ROC']
    }

with open(os.path.join(models_dir, "dl_models_config.json"), "w") as f:
    json.dump(config_dl, f, indent=4)
print("✅ Configurations DL sauvegardées")

# Vérification modèles sauvegardés
print("\n── Modèles DL sauvegardés ──")
for fname in sorted(os.listdir(models_dir)):
    if any(x in fname for x in ['cnn1d', 'ft_transformer']):
        size = os.path.getsize(os.path.join(models_dir, fname)) / (1024*1024)
        print(f"  {fname:<45} {size:.1f} MB")

print(f"\n── Tableau final ──")
print(df_dl.to_string(index=False))
print("\n🎉 Phase 3 complètement sauvegardée !")

✅ Tableau résultats DL sauvegardé
✅ Configurations DL sauvegardées

── Modèles DL sauvegardés ──
  cnn1d_ember.keras                             9.7 MB
  cnn1d_malmem.keras                            1.7 MB
  ft_transformer_ember.keras                    1.2 MB
  ft_transformer_malmem.keras                   0.4 MB

── Tableau final ──
         Model    Dataset  Threshold  Accuracy  Precision  Recall     F1  AUC-ROC
         CNN1D CIC-MalMem      0.397    1.0000     1.0000  1.0000 1.0000   1.0000
         CNN1D      EMBER      0.478    0.9514     0.9474  0.9558 0.9516   0.9889
FT-Transformer CIC-MalMem      0.100    0.9999     1.0000  0.9998 0.9999   1.0000
FT-Transformer      EMBER      0.414    0.9240     0.9227  0.9256 0.9242   0.9789

🎉 Phase 3 complètement sauvegardée !
